# PurushTradingAIAgents — full agent graph demo on Colab/Kaggle (no API keys, no cost)

Runs the complete pipeline on recorded/synthetic data: ML analysts → Verifier → regime → ensemble → Bull/Bear debate (local LLM) → trader → risk rules → PM report → paper fill.

The LLM runs **in-process** with llama.cpp on the free GPU (no Ollama server needed). Replace `YOUR_GITHUB_USER`.

> Paper research only. Not investment advice.

In [ ]:
!pip -q install "git+https://github.com/YOUR_GITHUB_USER/PurushTradingAIAgents.git#egg=ptaa"
!CMAKE_ARGS="-DGGML_CUDA=on" pip -q install llama-cpp-python  # CPU-only also works, slower
!pip -q install huggingface_hub
from huggingface_hub import hf_hub_download
GGUF = hf_hub_download('Qwen/Qwen3-4B-GGUF', 'Qwen3-4B-Q4_K_M.gguf')  # ~2.5 GB; or 'bartowski/Llama-3.2-3B-Instruct-GGUF'
print(GGUF)

In [ ]:
import os, pathlib
DATA = pathlib.Path('/content/ptaa-data'); DATA.mkdir(exist_ok=True)
os.environ.update(PTAA_DATA_DIR=str(DATA), PTAA_LLM_BACKEND='inprocess', PTAA_LLM_MODEL=GGUF)
!curl -sO https://raw.githubusercontent.com/YOUR_GITHUB_USER/PurushTradingAIAgents/main/config.yaml
from ptaa.settings import load_config, get_settings
cfg = load_config('config.yaml'); s = get_settings(); print(s.llm_backend, cfg.universe)

In [ ]:
# 1. Train a GBM on synthetic data so the technical analyst has a registered model (2 s). Use the training notebook for real data.
from ptaa.mlmodels.registry import ModelRegistry
from ptaa.training.pipeline import run_training, synthetic_candles
reg = ModelRegistry(DATA / 'models')
rep = run_training(cfg, reg, synthetic_candles, horizon='5d', universe='NIFTY50_LIQUID15', n_folds=4, min_train=3000)
rep.metrics

In [ ]:
# 2. Build the context (paper ledger + in-process LLM) and run the full graph for one symbol
from ptaa.graph.context import GraphContext
from ptaa.graph.graph import build_graph, run_once
from ptaa.graph.nodes.data import market_snapshot
from ptaa.ledger.db import Ledger, PaperBroker
from ptaa.llm.client import LLMClient
ledger = Ledger(DATA / 'ledger.db', cfg.fees, cfg.paper_book_inr)
ctx = GraphContext(cfg=cfg, settings=s, registry=reg, ledger=ledger, broker=PaperBroker(ledger), llm=LLMClient(s, cfg.llm))
ctx.news_items = []  # offline: no web discovery in the demo; set to None for live RSS/ddgs
snap = market_snapshot(synthetic_candles('RELIANCE'), synthetic_candles('NIFTY')['close'])
out = run_once(build_graph(ctx), ctx, 'RELIANCE', 'swing', snap)
for t in out['debate']: print(t.side.upper(), t.round, t.text)
print('THESIS', out['thesis'].narrative, out['thesis'].key_risks)
print('DECISION', out['decision'].action, out['decision'].quantity)
print(out['decision'].report)
print('llm calls', ctx.llm.stats.calls, 'failures', ctx.llm.stats.failures)

In [ ]:
# 3. Audit trail and ledger
for a in ledger.audit_for(out['run_id']): print(f"{a.node:<20} {a.duration_ms:8.1f} ms  {a.inputs_hash} -> {a.outputs_hash}")
print(ledger.positions()); print(ledger.equity())

## What to look for

* Every number in the debate and report exists in the verified signals; fabricated figures are stripped by the guard.
* `thesis.conviction` equals the ensemble score — the LLM cannot set it.
* Set `PTAA_LLM_BACKEND=none` and rerun: the decision is identical, only the prose disappears.